## K.Lab tiles computation
Get the gdf of GADM and the desiferd tiles grid from PostgreSQL and the the intersected tiles as a list from the country to the coastal level.



In [1]:
import geopandas as gpd
from sqlalchemy import create_engine, text

import pandas as pd
from shapely.ops import unary_union

In [ ]:
def load_vector_layer(db_name, user, password, host, port, table_name, schema='public', geom_col='geom'):
    """
    Connects to a PostGIS-enabled PostgreSQL database and loads a vector layer as a GeoDataFrame.
    
    Parameters:
    - db_name (str): Name of the PostgreSQL database.
    - user (str): Database username.
    - password (str): Database password.
    - host (str): Host address (e.g., 'localhost' or IP).
    - port (int): Port number (e.g., 5432).
    - table_name (str): Name of the table (vector layer) to load.
    - schema (str): Optional. Database schema containing the table (default is 'public').

    Returns:
    - gpd.GeoDataFrame: A GeoDataFrame containing the vector layer.
    """
    try:
        # Use pg8000 (pure Python driver)
        conn_str = f"postgresql+pg8000://{user}:{password}@{host}:{port}/{db_name}"
        engine = create_engine(conn_str)

        sql = text(f"SELECT * FROM {schema}.{table_name}")

        # Open a connection explicitly (SQLAlchemy 2.x requirement)
        with engine.connect() as conn:
            gdf = gpd.read_postgis(sql, conn, geom_col=geom_col)
        
        print(f"Successfully loaded {table_name} ({len(gdf)} features)")
        return gdf

    except Exception as e:
        print(f"Error loading vector layer: {e}")
        return None
    

def intersecting_tiles_list(country_gdf, grid_gdf):
    """
    Returns a list of tile_index values from the grid that intersect each country polygon.

    Parameters:
    - country_gdf: GeoDataFrame with a 'country_n' column and country polygons
    - grid_gdf: GeoDataFrame with a 'tile_index' column and grid cell polygons

    Returns:
    - pandas DataFrame with columns ['country', 'tile_indices']
      where 'tile_indices' is a list of tile_index values intersecting the polygon.
    """
    results = []
    geom_col = country_gdf.geometry.name

    for idx, feature in country_gdf.iterrows():
        geom = feature[geom_col]

        # Spatial filter: grid cells that intersect the geometry
        possible_tiles = grid_gdf[grid_gdf.intersects(geom)]

        # Extract intersecting tile indices
        tiles = possible_tiles["tile_index"].tolist()

        results.append({
            'country': feature.get('country_n'),
            'tile_indices': tiles
        })

    return pd.DataFrame(results)

def extract_country_coastlines(gadm_gdf, countries):
    """
    Extract true sea-facing coastlines for one or more countries
    from a GADM GeoDataFrame, excluding shared inland borders.

    Parameters
    ----------
    - gadm_gdf (geopandas.GeoDataFrame): GADM geometries with 'country_n' column for country names.
    - countries (str | list[str]): Country name or list of country names to extract coastlines for.

    Returns
    -------
    coast_gdf (geopandas.GeoDataFrame): GeoDataFrame containing only sea-facing coastlines 
    with a 'country_n' column for country names.
    """

    if isinstance(countries, str):
        countries = [countries]

    results = []

    for country in countries:
        target = gadm_gdf[gadm_gdf["country_n"] == country]
        if target.empty:
            continue

        target_union = unary_union(target.geometry)

        # Only use countries that touch the target as neighbors
        gadm_gdf["touches_target"] = gadm_gdf.geometry.touches(target_union)
        neighbors = gadm_gdf[gadm_gdf["touches_target"] & (gadm_gdf["country_n"] != country)]

        # Merge neighboring polygons and subtract them from the boundary
        others_union = unary_union(neighbors.geometry)
        coastline_geom = target_union.boundary.difference(others_union)
        
        # Create a temporary GeoDataFrame
        coast_gdf = gpd.GeoDataFrame(
            {"country_n": [country]}, geometry=[coastline_geom], crs=gadm_gdf.crs
        ).explode(index_parts=False).reset_index(drop=True)

        # Clean geometries
        coast_gdf = coast_gdf[coast_gdf.is_valid & (coast_gdf.length > 0)]
        results.append(coast_gdf)

    if not results:
        raise ValueError(f"No valid coastlines extracted for {countries}")
    # Combine and dissolve by country
    combined = gpd.GeoDataFrame(pd.concat(results, ignore_index=True), crs=gadm_gdf.crs)
    dissolved = combined.dissolve(by="country_n", as_index=False)

    return dissolved

In [ ]:
"""Load the data"""
# Load the GADM
gdf_gadm = load_vector_layer(
    db_name='',
    user='',
    password='',
    host='',
    port=,
    table_name='administrative_units_un_gadm_level0',
    schema=''
)

# Load the grid
gdf_grid = load_vector_layer(
    db_name='',
    user='',
    password='',
    host='',
    port=,
    table_name='global_klab_1d_tiles',
    schema=''
)


Successfully loaded administrative_units_un_gadm_level0 (318 features)
Successfully loaded global_klab_1d_tiles (50760 features)


In [ ]:
"""Country based tile listing"""
# Select the country
countries = ["Colombia", "Ecuador", "Peru"]
gdf_gadm_country = gdf_gadm[gdf_gadm['country_n'].isin(countries)]

# Get the tiles
df_result = intersecting_tiles_list(gdf_gadm_country, gdf_grid)

In [ ]:
"""Coastal based tile listing"""
countries = ["Kenya", "Somalia"] # Example

country_coastlines = extract_country_coastlines(gdf_gadm, countries)
df_result = intersecting_tiles_list(country_coastlines, gdf_grid)